# API Endpoints Test

This notebook calls the endpoint handler methods directly so we can verify they return API-like outputs before running the FastAPI server.


In [1]:
from __future__ import annotations
import pandas as pd

from pathlib import Path
import sys
import json

ROOT = Path.cwd().resolve()
if not (ROOT / 'core').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from fastapi.responses import JSONResponse, FileResponse
from starlette.responses import Response

from app.main import (
    health,
    crypto_return_service,
    forecast_horizon_endpoint,
    forecast_horizon_multi_endpoint,
    portfolio_recommend_endpoint,
    serve_frontend_root,
    serve_frontend_spa,
    HorizonRequest,
    MultiHorizonRequest,
    PortfolioRequest,
    _as_jsonable,
)
from app.schemas.crypto_return_service import CryptoReturnServiceRequest
from core.storage.coin_repository import get_coin_repository
from core.storage.market_data_repository import get_market_data_repository

coin_repo = get_coin_repository()
market_repo = get_market_data_repository()

available_symbols = sorted(coin_repo.list_symbols())
available_tickers = sorted(coin_repo.list_yahoo_tickers())

if not available_symbols:
    raise RuntimeError('No symbols found in Coins table.')
if not available_tickers:
    raise RuntimeError('No yahoo tickers found in Coins table.')

sample_symbol = 'BTC' if 'BTC' in available_symbols else available_symbols[0]
sample_multi_symbols = [s for s in ['BTC', 'ETH'] if s in available_symbols] or available_symbols[:2]
sample_portfolio_symbols = [s for s in ['BTC', 'ETH', 'ADA'] if s in available_symbols] or available_symbols[: min(3, len(available_symbols))]
sample_crypto_service_tickers = [t for t in ['BTC-USD', 'ETH-USD', 'ADA-USD'] if t in available_tickers] or available_tickers[: min(3, len(available_tickers))]

if len(sample_crypto_service_tickers) == 0:
    raise RuntimeError('Need at least one ticker for /crypto_return_service test.')

weights = [1.0 / len(sample_crypto_service_tickers)] * len(sample_crypto_service_tickers)
sample_assets = {ticker: weight for ticker, weight in zip(sample_crypto_service_tickers, weights)}

sample_features = market_repo.read_features(symbol=sample_symbol)
if sample_features.empty:
    raise RuntimeError(f'No feature rows found for symbol={sample_symbol}.')

sample_features['date'] = pd.to_datetime(sample_features['date'], errors='coerce')
sample_features = sample_features.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
sample_start_date = str(sample_features['date'].iloc[0].date())
sample_end_date = str(sample_features['date'].iloc[-1].date())

multi_feature_frames = {}
for symbol in sample_multi_symbols:
    df = market_repo.read_features(symbol=symbol)
    if df.empty:
        raise RuntimeError(f'No feature rows found for multi symbol={symbol}.')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    multi_feature_frames[symbol] = df

sample_multi_start_date = str(max(df['date'].iloc[0] for df in multi_feature_frames.values()).date())
sample_multi_end_date = str(min(df['date'].iloc[-1] for df in multi_feature_frames.values()).date())

portfolio_feature_frames = {}
for symbol in sample_portfolio_symbols:
    df = market_repo.read_features(symbol=symbol)
    if df.empty:
        raise RuntimeError(f'No feature rows found for portfolio symbol={symbol}.')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    portfolio_feature_frames[symbol] = df

sample_portfolio_start_date = str(max(df['date'].iloc[0] for df in portfolio_feature_frames.values()).date())


def dump_model(obj):
    if hasattr(obj, 'model_dump'):
        return obj.model_dump()
    if hasattr(obj, 'dict'):
        return obj.dict()
    return obj


def normalize_output(obj):
    if isinstance(obj, (dict, list, tuple)):
        return _as_jsonable(obj)
    if hasattr(obj, 'model_dump') or hasattr(obj, 'dict'):
        return _as_jsonable(dump_model(obj))
    if isinstance(obj, JSONResponse):
        body = obj.body.decode('utf-8') if hasattr(obj, 'body') else ''
        parsed = json.loads(body) if body else None
        return {
            'response_type': type(obj).__name__,
            'status_code': obj.status_code,
            'content': parsed,
        }
    if isinstance(obj, FileResponse):
        return {
            'response_type': type(obj).__name__,
            'status_code': obj.status_code,
            'path': str(obj.path),
            'media_type': obj.media_type,
            'filename': obj.filename,
        }
    if isinstance(obj, Response):
        body = obj.body.decode('utf-8') if hasattr(obj, 'body') else ''
        return {
            'response_type': type(obj).__name__,
            'status_code': obj.status_code,
            'body': body,
        }
    return _as_jsonable(obj)


def show_output(obj):
    payload = normalize_output(obj)
    print(json.dumps(payload, indent=2, default=str)[:12000])
    return payload


print('Project root:', ROOT)
print('Sample symbol:', sample_symbol)
print('Sample multi symbols:', sample_multi_symbols)
print('Sample portfolio symbols:', sample_portfolio_symbols)
print('Sample crypto service tickers:', sample_crypto_service_tickers)
print('Sample start/end:', sample_start_date, sample_end_date)
print('Sample multi start/end:', sample_multi_start_date, sample_multi_end_date)
print('Sample portfolio start:', sample_portfolio_start_date)


[explain] enabled=True provider=disabled client_loaded=False
Project root: C:\Users\sia\Desktop\capstone\src\Agentic-Crypto-Return-Service
Sample symbol: BTC
Sample multi symbols: ['BTC', 'ETH']
Sample portfolio symbols: ['BTC', 'ETH', 'ADA']
Sample crypto service tickers: ['BTC-USD', 'ETH-USD', 'ADA-USD']
Sample start/end: 2014-10-17 2026-03-15
Sample multi start/end: 2017-12-09 2026-03-15
Sample portfolio start: 2017-12-09


## `GET /health`

Calls `health()` directly and expects a JSON dict.

In [2]:
result = health()
health_payload = show_output(result)


{
  "status": "ok"
}


## `POST /crypto_return_service`

Calls `crypto_return_service(request)` directly with a valid `CryptoReturnServiceRequest`.

In [5]:
request = CryptoReturnServiceRequest(
    capital=1000.0,
    assets=sample_assets,
    horizon_days=14,
    n_scenarios=100,
    risk_tolerance=0.10,
)
result = crypto_return_service(request)
crypto_return_service_payload = show_output(result)


{
  "regime_matching": {
    "BTC-USD": {
      "current_window": {
        "ticker": "BTC-USD",
        "current_window_start_date": "2026-02-12",
        "current_window_end_date": "2026-03-13",
        "match_window_days": 30,
        "top_n": 10,
        "horizon_days": 14
      },
      "matches": [
        {
          "rank": 1,
          "candidate_window_idx": 2805,
          "window_end_index": 2835,
          "distance": 0.07505667209625244,
          "similarity": 0.9249433279037476,
          "window_start_index": 2805,
          "window_end_index_inclusive": 2834,
          "window_start_date": "2022-06-22",
          "window_end_date": "2022-07-21",
          "forward_end_index": 2848,
          "forward_end_date": "2022-08-04",
          "horizon_days": 14,
          "profit_pct": -0.023038222505520567,
          "max_drawdown_pct": -0.06494501451387713
        },
        {
          "rank": 2,
          "candidate_window_idx": 2806,
          "window_end_index": 2836,
 

## `POST /forecast/horizon`

Calls `forecast_horizon_endpoint(req)` directly with a valid `HorizonRequest`.

In [3]:
request = HorizonRequest(
    symbol=sample_symbol,
    start_date=sample_multi_start_date,
    horizon_days=14,
    engine='fast_regime_fixed',
    n_scenarios=500,
    alpha=0.05,
    seed=42,
    return_format='both',
    include_explanation=False,
)
result = await forecast_horizon_endpoint(request)
forecast_horizon_payload = show_output(result)


{
  "symbol": "BTC",
  "start_date": "2017-12-09",
  "end_date": "2017-12-29",
  "horizon_days": 14,
  "n_scenarios": 500,
  "alpha": 0.05,
  "engine": "fast_regime_fixed",
  "assumptions": {
    "engine": "fast_regime_fixed",
    "timeout_seconds": 12,
    "rate_limit_per_minute": 60
  },
  "summary": {
    "median_log": 0.08881712295247357,
    "median_simple": 0.09288077524058136,
    "mean_log": 0.08175241670992327,
    "mean_simple": 0.08518710237402916,
    "p05_log": -0.20252525375826103,
    "p05_simple": -0.18333414154447614,
    "p95_log": 0.3622741995204671,
    "p95_simple": 0.4365928010170472,
    "VaR_5_log": -0.20252525375826103,
    "VaR_5_simple": -0.18333414154447614,
    "CVaR_5_log": -0.272298869329669,
    "CVaR_5_simple": -0.23837339976036243
  },
  "risk": null,
  "metrics": null,
  "risk_curve_metrics": null,
  "explanation": null
}


## `POST /forecast/horizon/multi`

Calls `forecast_horizon_multi_endpoint(req)` directly with a valid `MultiHorizonRequest`.

In [4]:
request = MultiHorizonRequest(
    symbols=sample_multi_symbols,
    start_date=sample_portfolio_start_date,
    horizon_days=14,
    engine='fast_regime_fixed',
    n_scenarios=500,
    alpha=0.05,
    seed=42,
    return_format='both',
    include_explanation=False,
)
result = await forecast_horizon_multi_endpoint(request)
forecast_horizon_multi_payload = show_output(result)


{
  "results": [
    {
      "symbol": "BTC",
      "start_date": "2017-12-09",
      "end_date": "2017-12-29",
      "horizon_days": 14,
      "n_scenarios": 500,
      "alpha": 0.05,
      "engine": "fast_regime_fixed",
      "assumptions": {
        "engine": "fast_regime_fixed",
        "timeout_seconds": 20,
        "rate_limit_per_minute": 60
      },
      "summary": {
        "median_log": 0.08881712295247357,
        "median_simple": 0.09288077524058136,
        "mean_log": 0.08175241670992327,
        "mean_simple": 0.08518710237402916,
        "p05_log": -0.20252525375826103,
        "p05_simple": -0.18333414154447614,
        "p95_log": 0.3622741995204671,
        "p95_simple": 0.4365928010170472,
        "VaR_5_log": -0.20252525375826103,
        "VaR_5_simple": -0.18333414154447614,
        "CVaR_5_log": -0.272298869329669,
        "CVaR_5_simple": -0.23837339976036243
      },
      "risk": null,
      "metrics": null,
      "risk_curve_metrics": null,
      "explanation

## `POST /portfolio/recommend`

Calls `portfolio_recommend_endpoint(req)` directly with a valid `PortfolioRequest`.

In [6]:
request = PortfolioRequest(
    symbols=sample_portfolio_symbols,
    start_date=sample_start_date,
    horizon_days=14,
    engine='walkforward_ml',
    n_scenarios=300,
    seed=42,
    confidence_levels=[0.95],
    user_risk_tolerance=50,
    top_k=min(3, len(sample_portfolio_symbols)),
    max_weight=0.50,
    min_weight=0.00,
    allow_cash=True,
    include_explanation=False,
)
result = await portfolio_recommend_endpoint(request)
portfolio_recommend_payload = show_output(result)


[portfolio] start
[portfolio] engine=walkforward_ml symbols=['BTC', 'ETH', 'ADA']
[portfolio] loading asset data for BTC
[portfolio] BTC price_rows=4197 feature_rows=4167
[portfolio] loading asset data for ETH
[portfolio] ETH price_rows=3048 feature_rows=3018
[portfolio] loading asset data for ADA
[portfolio] ADA price_rows=3048 feature_rows=3018
[portfolio] model_types=['quantile_ml_walk_forward']
[portfolio] building config for model_type=quantile_ml_walk_forward
[portfolio] running pipeline for model_type=quantile_ml_walk_forward
[run_portfolio_pipeline] start
[run_portfolio_pipeline] symbols=['BTC', 'ETH', 'ADA']
[run_portfolio_pipeline] model_type=quantile_ml_walk_forward horizon_days=14 n_scenarios=300 seed=42
[run_portfolio_pipeline] processing symbol=BTC
[run_portfolio_pipeline] BTC price_rows=4197 feature_rows=4167
[run_portfolio_pipeline] BTC scenario generation start
[run_portfolio_pipeline] BTC scenario generation done in 83.89s; output_keys=['asset', 'distribution', 'summa

## `GET /`

Calls `serve_frontend_root()` directly. If the frontend build is missing this returns a JSON 404 payload; otherwise it returns a `FileResponse` that we normalize into a JSON-like dict.

In [7]:
result = serve_frontend_root()
frontend_root_payload = show_output(result)


{
  "response_type": "JSONResponse",
  "status_code": 404,
  "content": {
    "detail": "Frontend build not found. Run `npm run build` in crypto-risk-dashboard."
  }
}


## `GET /{full_path:path}`

Calls `serve_frontend_spa(full_path)` directly with a non-API path. As with the root route, file responses are normalized into a JSON-like dict.

In [8]:
result = serve_frontend_spa('dashboard/test-path')
frontend_catchall_payload = show_output(result)


{
  "response_type": "JSONResponse",
  "status_code": 404,
  "content": {
    "detail": "Frontend build not found. Run `npm run build` in crypto-risk-dashboard."
  }
}


## `Run API on Local Computer`

Step 1 : open `cmd`

Step 2 : Go to your path `cd path/to/your/project`

Step 3 : run this command:

```bash
uvicorn app.main:app --reload
```

Step 4 : open http://127.0.0.1:8000/docs